In [ ]:
import MetaTrader5 as mt5
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt

# Configuración inicial
csv_path = 'D:/iqrobot/btcusdt_data.csv'
model_path = 'D:/YOP/web_page_geosierra/lstm_price_prediction_model.h5'
seq_length = 50  # Debe coincidir con el modelo entrenado
error_threshold = 2.0  # 2% de margen de error aceptable

# Inicializar MetaTrader 5
if not mt5.initialize():
    print("Error al inicializar MetaTrader5:", mt5.last_error())
    quit()

# Conectar a la cuenta (asegúrate de usar tus credenciales correctas)
account = 1510191592  # Número de cuenta de FTMO
password = "8!*?9g6gq@W"  # Contraseña de la cuenta
server = "FTMO-Demo"  # Servidor de Meta en este caso FTMO

if not mt5.login(account, password=password, server=server):
    print("Error al iniciar sesión:", mt5.last_error())
    mt5.shutdown()
    quit()

# Lista de intervalos y temporalidades
intervalos = {
    '4h': mt5.TIMEFRAME_H4,  # 4 horas
    '1h': mt5.TIMEFRAME_H1,  # 1 hora
    '15m': mt5.TIMEFRAME_M15  # 15 minutos
}

# Fecha de fin (Fecha actual) y de inicio (6 meses antes)
end_date = datetime.now()
start_date = end_date - relativedelta(months=6)

# Obtener y guardar datos históricos en diferentes intervalos
for interval, timeframe in intervalos.items():
    rates = mt5.copy_rates_range("EURUSD", timeframe, start_date, end_date)

    if rates is None:
        print(f"Error al obtener datos para {interval}: {mt5.last_error()}")
        continue

    # Convertir a DataFrame de Pandas
    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')  # Convertir timestamps a fechas legibles

    # Guardar en CSV
    csv_filename = f"BTCUSD_{interval}_data.csv"
    df.to_csv(csv_filename, index=False)

    # Análisis de datos con el modelo LSTM
    try:
        # Carga y preparación de datos
        data = pd.read_csv(csv_filename)
        print(f"Datos cargados exitosamente para {interval}. Registros:", len(data))
        
        # Validar columnas requeridas
        required_columns = ['close', 'time']
        if not all(col in data.columns for col in required_columns):
            raise ValueError(f"El CSV debe contener: {', '.join(required_columns)}")

        # Convertir time y ordenar
        data['time'] = pd.to_datetime(data['time'])
        data.sort_values('time', inplace=True)

        # Preprocesamiento de datos
        scaler = MinMaxScaler(feature_range=(0, 1))
        data_numeric = data.select_dtypes(include=[np.number])
        data_scaled = scaler.fit_transform(data_numeric)

        # Creación de secuencias completas
        def create_sequences_full(data, window_size):
            X, y = [], []
            for i in range(len(data) - window_size):
                X.append(data[i:i+window_size])
                y.append(data[i+window_size, 3])  # close es la cuarta columna
            return np.array(X), np.array(y)

        X_full, y_full = create_sequences_full(data_scaled, seq_length)
        X_full = X_full.reshape((X_full.shape[0], seq_length, data_numeric.shape[1]))

        # Carga del modelo
        model = load_model(model_path)
        print("Modelo LSTM cargado exitosamente")

        # Predicción de todas las velas
        y_pred_scaled = model.predict(X_full)

        # Inversión de normalización
        def inverse_transform(scaler, data, col_index=0):
            dummy = np.zeros((len(data), data_numeric.shape[1]))
            dummy[:, col_index] = data
            return scaler.inverse_transform(dummy)[:, col_index]

        y_real = inverse_transform(scaler, y_full, col_index=3)  # close es la cuarta columna
        y_pred = inverse_transform(scaler, y_pred_scaled.ravel(), col_index=3)

        # Creación del DataFrame de resultados
        predictions_df = pd.DataFrame({
            'Fecha': data['time'].iloc[seq_length:].values,
            'Real': y_real,
            'Prediccion': y_pred
        })

        # Visualización gráfica de las últimas 16 velas + pronóstico
        plt.figure(figsize=(14, 7))
        plt.plot(predictions_df['Fecha'].iloc[-16:], predictions_df['Real'].iloc[-16:], label='Precio Real', color='blue', alpha=0.6)
        plt.plot(predictions_df['Fecha'].iloc[-16:], predictions_df['Prediccion'].iloc[-16:], label='Predicción', color='orange', alpha=0.6)

        # Predicción de las próximas 4 velas
        last_sequence = data_scaled[-seq_length:]
        future_predictions = []

        for _ in range(4):
            input_seq = last_sequence.reshape(1, seq_length, data_numeric.shape[1])
            pred = model.predict(input_seq)[0,0]
            pred = np.clip(pred, 0, 1)  # Asegurar valores entre 0 y 1
            new_row = np.hstack((pred, last_sequence[-1, 1:]))
            last_sequence = np.vstack([last_sequence[1:], new_row])
            future_predictions.append(pred)

        # Inversión de normalización para el pronóstico futuro
        future_prices = inverse_transform(scaler, np.array(future_predictions), col_index=3)
        future_dates = pd.date_range(start=predictions_df['Fecha'].iloc[-1], periods=5, freq='4H')[1:]

        # Graficar pronóstico futuro
        plt.plot(future_dates, future_prices, label='Pronóstico Futuro', color='red', linestyle='dotted')

        plt.title('Predicciones y Pronóstico Futuro (Últimas 16 velas)', fontsize=14)
        plt.xlabel('Fecha')
        plt.ylabel('Precio (USD)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

        # Guardar resultados
        predictions_df.to_csv(f'reporte_precision_detallado_{interval}.csv', index=False)
        print(f"Reporte de precisión detallado guardado para {interval} en 'reporte_precision_detallado_{interval}.csv'")
    except Exception as e:
        print(f"Error al analizar datos para {interval}: {str(e)}")
    print(f"Datos de {interval} guardados en {csv_filename}. Iniciando análisis de datos...")

# Cerrar la conexión MetaTrader 5
mt5.shutdown()
